Dependencies

In [54]:
import json
import requests
import time
import os
import re
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from bs4 import BeautifulSoup
from dotenv import load_dotenv

Global Variable

In [ ]:
# Memuat variabel dari file .env
load_dotenv()

# URL API
url_main = os.getenv("URL_MAIN")
url_login = os.getenv("URL_LOGIN")
url_article_list = os.getenv("URL_ARTICLE_LIST")
url_article_content = os.getenv("URL_ARTICLE_CONTENT")
# print(url_article_content)

# Credentials dan Headers
username = os.getenv("MDL_USERNAME")
password = os.getenv("MDL_PASSWORD")
org_code = os.getenv("ORG_CODE")

# print(f"Username: {username}, Password: {password}")

credentials = {
    "username":username,
    "password":password,
    "type":"sso",
    "org_code":org_code,
    "app_version":"1.0",
    "device":"web",
    "platform":"saas"
}

# Lokasi file raw data
file_list_path=os.getenv("FILE_LIST_PATH")
file_content_path=os.getenv("FILE_CONTENT_PATH") 
file_ready_data_path=os.getenv("FILE_READY_DATA_PATH")

# print(file_ready_data_path)

./data/scraping/ready_data.csv


Login

In [35]:
response_login = requests.post(url_login, json=credentials)
if response_login.status_code == 200:
    data = response_login.json()
    token = data["data"]["token"]
    print(f"Token didapat: {token}")
else:
    print(f"Token gagal didapat: {response_login.status_code}")
    print(f"Alasan: {response_login.text}")

headers = {
    "accept": "application/json, text/plain, */*",
        "Authorization": f"Bearer {token}",
        "dnt": "1",
        "origin": url_main,
        "priority": "u=1, i",
        "referer": f"{url_main}/",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "Windows",
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-site",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"
}

Token didapat: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpZCI6MTYzMjc0LCJ1c2VydGFnIjoidmViZXJpYXBhbmphaXRhbiIsInVzZXJuYW1lIjoiOTgwMDA2IiwiZW1haWwiOiJ2ZWJlcmlhLnBhbmphaXRhbkB0ZWxrb20uY28uaWQiLCJwaG9uZSI6IjA4MjE3NTEwNjI2MiIsImlzX3RlbGtvbV9ncm91cCI6dHJ1ZSwiY3VycmVudF9wcm9ncmFtIjoxLCJwcm9ncmFtcyI6W3siaWQiOjEsInRpdGxlIjoibXlEaWdpTGVhcm4iLCJzbHVnIjoidGVsa29tbWRsIiwib3JnX2NvZGUiOiJrOVd4MDNIV3RoMmJmZHpPMlRZOCIsIm9yZ2hvbWVwYWdlX2lkIjpudWxsLCJqb3VybmV5aG9tZXBhZ2VfaWQiOm51bGwsImlzX21haW5fZG9tYWluIjp0cnVlfV0sInByb2dyYW0iOnsiaWQiOjEsInRpdGxlIjoibXlEaWdpTGVhcm4iLCJzbHVnIjoidGVsa29tbWRsIiwib3JnX2NvZGUiOiJrOVd4MDNIV3RoMmJmZHpPMlRZOCIsIm9yZ2hvbWVwYWdlX2lkIjpudWxsLCJqb3VybmV5aG9tZXBhZ2VfaWQiOm51bGwsImlzX21haW5fZG9tYWluIjp0cnVlfSwic291cmNlIjoic2FhcyIsInNjb3BlIjpbImxlYXJuZXIiXSwiY3VycmVudF9jb21wYW55IjoxNSwiY29tcGFueV9pZHMiOlsxNV0sImNvbXBhbmllcyI6W3siaWQiOjE1LCJuYW1lIjoiUFQuIFRlbGtvbSBJbmRvbmVzaWEgKFBlcnNlcm8pIFRiay4iLCJwaWN0dXJlIjoiaHR0cHM6Ly9kMXVxdHRibnJjd2V1OC5jbG91ZGZyb250Lm5ldC9jbXMtdjIvY29tcGFueS9hdmF0

Scraping Article List Newer

In [36]:
id_inc_highest_local = 0 
local_article = []

# Ambil Checkpoint
if os.path.exists(file_list_path):
    with open(file_list_path, "r", encoding="utf-8") as article_list_file:
        try:
            print("tes")
            local_article = json.load(article_list_file)
            if len(local_article) > 0:
                id_inc_highest_local = max([item['id_inc'] for item in local_article])
                print(f"📌 Resume dari id_inc terakhir: {id_inc_highest_local}")
            else:
                print("📌 File ada, tapi isinya list kosong. mulai dari 0")
                
        except json.JSONDecodeError:
            print("File lokal kosong atau rusak, mulai dari nol.")
            local_article = []

print("\nMemulai pencarian artikel baru...")

set_id_local = set([item["id_inc"] for item in local_article])
newer_article = []
max_scrap = 10
overlap_limit = 10
overlap_counter = 0
i = 1
page = 1
limit = 10

stop_scrap = False

print("Mulai menjelajahi data terbaru - lokal")

while not stop_scrap:
    print(f"\n📄 Membuka Halaman {page}...")
    
    params_list = {
        "page": page,
        "limit": limit
    }
    
    response_list = requests.get(url_article_list, params=params_list, headers=headers)


    if response_list.status_code != 200:
        print(f"❌ Gagal buka halaman {page}. Status: {response_list.status_code}")
        break

    # response_list_json = response_list_list.json()
    # data = response_list_json.get("data", {})
    # data_list = data.get("contents", {})
    data = response_list.json().get("data", {})
    data_list = data.get("contents", [])

    if not data_list:
        print("🛑 Halaman kosong! Kita sudah mencapai ujung dunia database.")
        break

    for article in data_list:
        id_inc_current = article["id_inc"]

        # SKENARIO A: Artikel SUDAH ADA di lokal
        if id_inc_current in set_id_local:
            overlap_counter += 1

            if overlap_counter >= overlap_limit:
                print(f"🛑 Menyentuh dasar data lama ({overlap_limit} beruntun). Tarik rem darurat!")
                stop_scrap = True # Beri sinyal agar Loop Luar juga ikut berhenti
                break # Hentikan Loop Dalam
            continue

        # SKENARIO B: Artikel BARU atau GAP
        overlap_counter = 0

        newer_article.append({
            'id_inc': id_inc_current,
            'id': article['id'],
            'slug': article['slug']
        })
        print(f"➕ Dapat artikel (Baru/Gap): {article['slug']}")

        i += 1

        # SKENARIO C: KUOTA HABIS
        if i > max_scrap:
            print(f"🛑 Kuota habis ({max_scrap}). Sisa gap dilanjut besok!")
            stop_scrap = True
            break
    
    page += 1

print("Menggabungkan dan membersihkan data...")

# Merge Data
dirty_data_merge = local_article + newer_article

# Deduplicate (Ubah jadi dictionary agar otomatis unik)
data_unique_dict = {item['id_inc']: item for item in dirty_data_merge}

# Kembalikan lagi wujudnya menjadi List
data_final = list(data_unique_dict.values())

# Sorting (Terbaru di Atas, Terlama di Bawah)
data_final.sort(key=lambda artikel: artikel['id_inc'], reverse=True)

# Simpan kembali ke lokal
with open(file_list_path, "w", encoding="utf-8") as f:
    json.dump(data_final, f, indent=4, ensure_ascii=False)

print(f"data lokal: {len(local_article)}")
print(f"data baru: {len(newer_article)}")

print(f"🎉 Sukses! Tersimpan {len(data_final)} artikel yang sudah rapi dan terurut.")

tes
📌 Resume dari id_inc terakhir: 18412

Memulai pencarian artikel baru...
Mulai menjelajahi data terbaru - lokal

📄 Membuka Halaman 1...
➕ Dapat artikel (Baru/Gap): dashboard-sudah-di-index-mengapa-masih-lambat-55420e3ed370
➕ Dapat artikel (Baru/Gap): pengadaan-barang-jasa-pemerintah-460d0993f0a7

📄 Membuka Halaman 2...
🛑 Menyentuh dasar data lama (10 beruntun). Tarik rem darurat!
Menggabungkan dan membersihkan data...
data lokal: 159
data baru: 2
🎉 Sukses! Tersimpan 161 artikel yang sudah rapi dan terurut.


Scraping Article List Older

In [37]:
local_article = []

if os.path.exists(file_list_path):
    with open(file_list_path, "r", encoding="utf-8") as f:
        try:
            local_article = json.load(f)
        except Exception as e:
            print("⚠️ File lokal ada tapi teksnya kosong melompong. Menganggap database 0.")
            local_article = []

# print(len(local_article))
if len(local_article) == 0:
    print("Database lokalmu masih kosong! Gunakan script 'Tarik Data Baru' saja mulai dari halaman 1.")
    # exit() # Hentikan script backfill
else: 
    print("cek artikel jadul")
    
    older_article = []
    set_id_local = set([item['id_inc'] for item in local_article])

    limit = 10
    max_scrap = 10
    i = 1
    estimation = len(local_article) // 10
    page = 1 if (estimation - 2) <= 0 else (estimation - 2)
    max_page = 20 + page 
    visited_page = 1
    stop_scrap = False

    # Mulai scraping
    while not stop_scrap and page <= max_page:
        
        params_list = {
            "page": page,
            "limit": limit
        }
        response_list = requests.get(url_article_list, params=params_list, headers=headers)
        data = response_list.json().get('data', [])
        data_list = data.get("contents", {})
        
        # JIKA MENTOK UJUNG DATABASE SERVER
        if not data_list:
            print("🛑 Halaman kosong! Kita sudah mencapai zaman purba (artikel pertama di server).")
            break

        # jika artikel jadul
        print(f"⛏️ Mulai menggali di Halaman {page}...")
        
        for article in data_list:
            id_inc_current = article["id_inc"]

            if id_inc_current not in set_id_local:
                older_article.append(
                    {
                        "id_inc": article["id_inc"],
                        "id": article["id"],
                        "slug": article["slug"]
                    }
                )

                print(f"➕ Dapat fosil article: {article['slug']}")

                i += 1
                
                if i > max_scrap:
                    print(f"🛑 Kuota habis ({max_scrap}). Sisa fosil dilanjut besok!")
                    stop_scrap = True
                    break
                
        page += 1
        time.sleep(5)

    print(f"\n🎉 Berhasil menggali {len(older_article)} article masa lalu!")

    # --- ASUMSI VARIABEL ---
    # local_article = Berisi list dari file JSON lama
    # newer_article = Berisi list artikel baru dari API
    # older_article = Berisi list artikel lama dari proses backfilling

    print("Menggabungkan dan membersihkan data...")

    # Merge Data
    dirty_data_merge = local_article + older_article

    # Deduplicate (Ubah jadi dictionary agar otomatis unik)
    data_unique_dict = {item['id_inc']: item for item in dirty_data_merge}

    # Kembalikan lagi wujudnya menjadi List
    data_final = list(data_unique_dict.values())

    # Sorting (Terbaru di Atas, Terlama di Bawah)
    data_final.sort(key=lambda artikel: artikel['id_inc'], reverse=True)

    # Simpan kembali ke lokal
    with open(file_list_path, "w", encoding="utf-8") as f:
        json.dump(data_final, f, indent=4, ensure_ascii=False)

    print(f"data lokal: {len(local_article)}")
    print(f"data jadul: {len(older_article)}")

    print(f"🎉 Sukses! Tersimpan {len(data_final)} artikel yang sudah rapi dan terurut.")

cek artikel jadul
⛏️ Mulai menggali di Halaman 14...
⛏️ Mulai menggali di Halaman 15...
⛏️ Mulai menggali di Halaman 16...
⛏️ Mulai menggali di Halaman 17...
⛏️ Mulai menggali di Halaman 18...
➕ Dapat fosil article: 8-powerfull-project-management-processes-3ae886cfbe7b
➕ Dapat fosil article: mapping-the-culinary-ecosystem-around-universitas-gadjah-mada-a-cluster-based-exploration-of-local-food-variations-fe9cd55e4b04
➕ Dapat fosil article: penguatan-posisi-telkom-regional-ii-dalam-pemenangan-market-government-kajian-analisa-market-landscape-strategi-marketing-dan-champion-product-government-customer-f0d8907b9e5e
➕ Dapat fosil article: book-resume-quiet-the-power-of-introverts-a-summary-of-strength-in-a-world-that-never-stops-talking-247c9dd09c92
➕ Dapat fosil article: strategic-planning-vs-tactical-planning-vs-operational-planning-3a413b99f7a9
➕ Dapat fosil article: data-inaproc-2026-8a36ad664995
➕ Dapat fosil article: pengelolaan-project-charter-dalam-manajemen-proyek-542975d77eac
➕ D

Scraping Article Content

In [41]:
# Mengambil konten setiap artikel dari daftar artikel
if os.path.exists(file_list_path):
    with open(file_list_path, "r", encoding="utf-8") as f:
        try:
            local_list_article = json.load(f)
        except Exception as e:
            print("⚠️ File lokal ada tapi teksnya kosong melompong. Menganggap database 0.")
            local_list_article = []

if len(local_list_article) == 0:
    print("File list kosong")
else:   
    print("ambil konten")

    if os.path.exists(file_content_path):
        with open(file_content_path, 'r', encoding="utf-8") as f:
            try:
                local_content_article = json.load(f)
            except Exception as e:
                print("⚠️ File lokal ada tapi teksnya kosong melompong. Menganggap database 0.")
                local_list_article = []

    content_article = []
    set_id_local = set([item['id_inc'] for item in local_content_article])

    max_scrap = len(newer_article) + len(older_article) + 10
    i = 1
    stop_scrap = False

    
    for article in local_list_article:
        id_inc_current = article["id_inc"]
        slug = article["slug"]

        if id_inc_current in set_id_local:
            # print(f"skip {id_inc_current}")
            continue

        try:
            url_article_content_final = f"{url_article_content.rstrip('/')}/{slug}"
            print(f"url konten: {url_article_content}")
            print(f"url final: {url_article_content_final}")
            response_artikel = requests.get(url_article_content_final, headers=headers)
            
            if response_artikel.status_code != 200:
                print(f"❌ Gagal mengambil {slug} (Status: {response_artikel.status_code})")
                time.sleep(5)
                continue

            data = response_artikel.json().get("data", {})
            
            # Cek apakah datanya benar-benar ada
            if not data:
                print(f"⚠️ Melewati {slug}: Isinya kosong dari server.")
                continue 
            
            content_article.append(
                {
                    "id_inc": data.get("id_inc"),
                    "id": data.get("id"),
                    "slug": data.get("slug", slug),
                    "title": data.get("title", "Tanpa Judul"), 
                    "content": data.get("article", ""),        "tags": data.get("tags", []),
                }
            )
        
            print(f"✅ Sukses menarik ({i}/{max_scrap}): {slug}")
            print(id_inc_current)

            i += 1

            if i > max_scrap:
                print(f"🛑 Kuota habis ({max_scrap}). Sisa gap dilanjut besok!")
                stop_scrap = True
                break

        except Exception as e:
            print(f"⚠️ Error tak terduga pada {slug}: {e}")

        time.sleep(5)

    print(f"Berhasil mengambil {len(content_article)} konten")

    if len(content_article) > 0:
        dirty_data_merge = local_content_article + content_article
        data_unique_dict = {item["id_inc"]: item for item in dirty_data_merge}
        data_final = list(data_unique_dict.values())
        data_final.sort(key=lambda article: article["id_inc"], reverse=True)

        with open(file_content_path, "w", encoding="utf-8") as f:
            json.dump(data_final, f, indent=4, ensure_ascii=False)

        print(f"data konten lokal: {len(local_content_article)}")
        print(f"data konten: {len(content_article)}")
        print(f"\n🎉 Selesai! {len(data_final)} konten telah ditambahkan")
    else:
        print("✅ Semua list artikel sudah memiliki konten. Tidak ada yang didownload.")

ambil konten
url konten: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug
url final: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug/8-powerfull-project-management-processes-3ae886cfbe7b
✅ Sukses menarik (1/22): 8-powerfull-project-management-processes-3ae886cfbe7b
18227
url konten: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug
url final: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug/mapping-the-culinary-ecosystem-around-universitas-gadjah-mada-a-cluster-based-exploration-of-local-food-variations-fe9cd55e4b04
✅ Sukses menarik (2/22): mapping-the-culinary-ecosystem-around-universitas-gadjah-mada-a-cluster-based-exploration-of-local-food-variations-fe9cd55e4b04
18226
url konten: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug
url final: https://gateway.mydigilearn.id/social/api/v1/digifeed/slug/penguatan-posisi-telkom-regional-ii-dalam-pemenangan-market-government-kajian-analisa-market-landscape-strategi-marketing-dan-champion-pro

Preprocessing and Getting The Data Ready

In [56]:
with open(file_content_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

def clear_html(html_tag):
    # Mengupas semua tag HTML dan menyisakan teks murni
    if not html_tag:
        return ""
    return BeautifulSoup(html_tag, "html.parser").get_text(separator=" ")

df['clean_content'] = df['content'].apply(clear_html)
df['tags_string'] = df['tags'].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df['complete_data'] = df['title'] + " " + df['tags_string'] + " " + df['clean_content']

def text_normalization(teks):
    teks = teks.lower() # Huruf kecil semua
    teks = re.sub(r'[^a-z0-9\s]', ' ', teks) # Buang karakter aneh/tanda baca
    teks = re.sub(r'\s+', ' ', teks).strip() # Rapikan spasi ganda
    return teks

df['ready_data'] = df['complete_data'].apply(text_normalization)
df_ready = df[['id_inc', 'id', 'slug', 'title', 'ready_data']]
df_ready = df_ready.dropna(subset=['ready_data'])

# Intip hasilnya
df_ready.info()
print(df_ready.head())
df_ready.to_csv(file_ready_data_path, index=False)

<class 'pandas.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_inc      171 non-null    int64
 1   id          171 non-null    str  
 2   slug        171 non-null    str  
 3   title       171 non-null    str  
 4   ready_data  171 non-null    str  
dtypes: int64(1), str(4)
memory usage: 6.8 KB
   id_inc                                    id  \
0   18414  3f00392d-dd75-42aa-a4ad-1b31725a7f58   
1   18413  08af95e8-8619-42dd-8748-2a425ff5046c   
2   18412  9a50e694-fa81-4aa0-8a87-24ca348d1a7c   
3   18411  fd77a1aa-c70e-46a7-a4a4-47d5a984a6de   
4   18410  3de46395-7299-4603-8384-cf5b4186178a   

                                                slug  \
0  dashboard-sudah-di-index-mengapa-masih-lambat-...   
1      pengadaan-barang-jasa-pemerintah-460d0993f0a7   
2                  merketing-management-070ca19bd8f9   
3  customer-profiling-segmentation-and-sales-pred...   
4

In [43]:
print("🚀 Memulai Vektorisasi Otomatis via ChromaDB...")

client = chromadb.PersistentClient(path="./chroma_db")
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)
collection = client.get_or_create_collection(
    name="article_db",
    embedding_function=embedder
)

data_in_db = collection.get(include=[])
set_id_db = set(data_in_db["ids"])
print(f"📦 Saat ini ada {len(set_id_db)} artikel di dalam ChromaDB.")

print("📊 Menyiapkan Data CSV...")
df_ready = pd.read_csv("ready_data.csv")
df_ready = df_ready.dropna(subset=['ready_data']) 

df_new = df_ready[~df_ready["id"].isin(set_id_db)]
if not df_new.empty:
    print(f"🔍 Menemukan {len(df_new)} artikel baru yang belum di-vektorisasi.")
    
    daftar_id = df_new["id"].tolist()
    daftar_dokumen = df_new["ready_data"].tolist()
    daftar_metadata = df_new[["id_inc", "slug", "title"]].to_dict(orient="records")

    collection.add(
        ids=daftar_id,
        documents=daftar_dokumen,
        metadatas=daftar_metadata
    )

    print(f"🎉 Selesai! {len(df_new)} artikel baru aman di ChromaDB.")
else:
    print("✨ Tidak ada artikel baru. Semua data sudah sinkron!")

print(f"🎉 Total data di ChromaDB: {collection.count()} artikel.")

🚀 Memulai Vektorisasi Otomatis via ChromaDB...
📦 Saat ini ada 160 artikel di dalam ChromaDB.
📊 Menyiapkan Data CSV...
🔍 Menemukan 11 artikel baru yang belum di-vektorisasi.
🎉 Selesai! 11 artikel baru aman di ChromaDB.
🎉 Total data di ChromaDB: 171 artikel.
